# Homework 2
#### EE-556 Mathematics of Data - Fall 2026

This homework covers Lectures 8-12. Please take a look at the material for the context and notation.

In this homework we will study minimax problems. We will begin with some theoretical analysis and in a second part you will implement a Wasserstein Generative Adversarial Network (WGAN). 

These notebooks should expose you to the fundamentals of GAN training at a basic level, as well as some of the theory behind it.

<div class="alert alert-info">
  ℹ️ <strong>Information on group based work:</strong>
</div>

- Only one group member needs to submit all five completed notebooks on behalf of the group.
- You may seek help outside your group, but each group must write its own answers.
- If the work was <span style="color: red;">unevenly distributed</span> among group members, let the TAs know.
- Only one member of the group is allowed to use AI. We will require sharing the conversation history with the AI in the form of a public link. If you use multiple conversations across the same or multiple tools please share all of them. Name the person in your group who is allowed to use AI. We encourage you to use the AI to help you understand the material, but we ask you to write the code and theory solutions by yourself. Part 5 permits AI usage as specified in its instructions.

<div style="border: 1px solid #f00; background-color: #fdd; padding: 10px; border-radius: 5px;">
  ⚠️ Do not forget: List each group member's name and SCIPER number.
</div>


Person 1 **Name**: Francesco Foglia || Person 1 **SCIPER**: 425511


Person 2 **Name**: Antonio Angelo Polignano|| Person 2 **SCIPER**: 436345




<div style="border: 1px solid #0a0; background-color: #dfd; padding: 10px; border-radius: 5px;">
  📓 Feedback on AI use: Please use the following cell to provide feedback on the AI use in this notebook.
  
  For example, how useful were the tools to you? Which tools did you use? Did you feel like they helped you understand the material better?
</div>

We provide an extract of the conversation, as claude cowork did not allow me to share a chat link

## 1. Minimax problems - 65 points

### 1.1 Theoretical recap: stationary points and convergence in minmax games - 30 points

Consider a stylized function $f:\mathbb{R}^4 \to \mathbb{R}$ with variables $x_1,x_2,y_1,y_2$, we denote by
$x=(x_1,x_2) \in \mathbb{R}^2$, $y=(y_1,y_2) \in \mathbb{R}^2$, the function has the form of
$
f(x,y) = (a x - b)^\top (a y - c), \quad a \neq 0 \in \mathbb{R}, \; b,c \in \mathbb{R}^2.
$


__(1)__ (5 points) Write down the first-order stationary points of $f$, and
        classify them as local minimum, local maximum, or saddle point by
        inspecting its Hessian.
        
**HINT:** If matrix $E, F, G, H$ commute with each other (for example, when they are all scalar multiples of the same identity matrix $I$), then

$$
\det
\begin{bmatrix}
E & F\\
G & H
\end{bmatrix}
=
\det(EH - FG).
$$


We first expand $f$ in terms of inner products:
$$
f(x,y) = a^2 \langle x, y \rangle - a \langle x, c \rangle - a \langle y, b \rangle + \langle b, c \rangle .
$$
Then
$$
\begin{align*}
\nabla f(x,y) &= a^2 \begin{pmatrix} y \\ x \end{pmatrix} - a \begin{pmatrix} c \\ b \end{pmatrix} = 0 \\ \\
\implies \begin{pmatrix} x^\star \\ y^\star \end{pmatrix} &= \begin{pmatrix} b/a \\ c/a \end{pmatrix}
\end{align*}
$$
which is the unique first-order stationary point, since $a \neq 0$. The Hessian is constant:
$$
\nabla^2 f = a^2 \begin{bmatrix} 0 & I \\ I & 0 \end{bmatrix}, \qquad I \in \mathbb{R}^{2 \times 2}.
$$
Its characteristic polynomial is
$$
\begin{align*}
\det\left(\nabla^2 f - \lambda I_4\right)
&= \det \begin{bmatrix} -\lambda I & a^2 I \\ a^2 I & -\lambda I \end{bmatrix} \\
&= \det\left(\lambda^2 I - a^4 I\right) \\
&= \left(\lambda^2 - a^4\right)^2 \\ \\
\implies \lambda_{1,2} &= \pm a^2
\end{align*}
$$
where the second equality is the hint, which applies because all blocks are scalar multiples of $I$ and therefore commute. Each eigenvalue has multiplicity $2$. Since $a \neq 0$, the Hessian has one strictly positive and one strictly negative eigenvalue, so it is indefinite and $(x^\star, y^\star)$ is a saddle point.

__(2)__ (5 points) Find the solution $(x^\star,y^\star)$ to the
        minimax problem $\min_x \max_y f(x, y)$. You can quantify the solution
        by using the following saddle point inequality:  $f(x^\star, y^\star)
        \geq f(x^\star, y)$ and $f(x^\star, y^\star) \leq f(x, y^\star)$, for
        all $x, y$.
        
**HINT:** $(x^\star, y^\star)$ can only be one of the critical points you found in (1), just evaluate $f$ at every place in the inequalities to check the optimality!




We check that the stationary point $(x^\star, y^\star) = (b/a, c/a)$ found in (1) satisfies equality case for the saddle point inequalities, that guarantee optimality. Since $a x^\star - b = 0$ and $a y^\star - c = 0$,
$$
\begin{align*}
f(x^\star, y) &= (a x^\star - b)^\top (a y - c) = 0 \quad \forall y, \\
f(x, y^\star) &= (a x - b)^\top (a y^\star - c) = 0 \quad \forall x, \\
f(x^\star, y^\star) &= 0 \\ \\
\implies f(x^\star, y) = f(x^\star, y^\star) &= f(x, y^\star) \quad \forall x, y
\end{align*}
$$
so both inequalities hold with equality. Hence $(x^\star, y^\star) = (b/a, c/a)$ is the solution of the minimax problem, and
$$
\min_x \max_y f(x,y) = f(x^\star, y^\star) = 0 .
$$

__(3)__ (10 points) One possible attempt at finding this solution via iterative
        first-order methods is to perform gradient updates on the variables $x$
        and $y$. More precisely for $\gamma > 0$ consider the simultaneous gradient
        descent/ascent updates
        
\begin{equation}
    x^{k+1} = x^k - \gamma \nabla_x f(x^k, y^k), \qquad
    y^{k+1} = y^k + \gamma \nabla_y f(x^k, y^k) \nonumber
\end{equation}
Show that the sequence of iterates $\{x^k, y^k \}_{k=0}^\infty$ starting
from any point $(x^0, y^0) \neq (x^\star, y^\star)$ diverges, for any $\gamma > 0$.
Find the rate at which the distance from 
$(x^\star,y^\star)$ to the sequence $\{x^k, y^k \}$ grows as the number of iterations $k$ increases.

**HINT:** Define $d_k^2=||(x^k,y^k)-(x^\star, y^\star)||_2^2$ as the sequence of squared distances to the optimum. If you find a formula for how $d_{k+1}$ depends on $d_k$ using the exact gradient updates for our $f$, you can easily argue for the divergence and the rate.

Using $\nabla_x f = a(ay - c)$, $\nabla_y f = a(ax - b)$ and $(x^\star, y^\star) = (b/a, c/a)$ from (2), we get
$$
\begin{align*}
d_{k+1}^2 &= \left\| \begin{pmatrix} x^k \\ y^k \end{pmatrix} + a^2\gamma \begin{pmatrix} -y^k + c/a \\ x^k - b/a \end{pmatrix} - \begin{pmatrix} b/a \\ c/a \end{pmatrix} \right\|_2^2 \\
&\overset{\text{Pythagoras}}{=} \left\|x^k - b/a + a^2\gamma(-y^k + c/a)\right\|_2^2 + \left\|y^k - c/a + a^2\gamma(x^k - b/a)\right\|_2^2 \\
&\overset{(i)}{=} \left\|x^k - b/a\right\|_2^2 (1 + a^4\gamma^2) + \left\|y^k - c/a\right\|_2^2 (1 + a^4\gamma^2) \\
&\qquad + 2\left\langle x^k - b/a,\, a^2\gamma(-y^k + c/a) \right\rangle + 2\left\langle y^k - c/a,\, a^2\gamma(x^k - b/a) \right\rangle \\
&\overset{(ii)}{=} d_k^2 (1 + a^4\gamma^2)
\end{align*}
$$
where (i) expands the squared norms as inner products, and in (ii) the two cross terms cancel, being opposite to each other, while
$$
d_k^2 = \left\| \begin{pmatrix} x^k \\ y^k \end{pmatrix} - \begin{pmatrix} b/a \\ c/a \end{pmatrix} \right\|_2^2 \overset{\text{Pythagoras}}{=} \left\|x^k - b/a\right\|_2^2 + \left\|y^k - c/a\right\|_2^2 .
$$
Therefore
$$
d_k^2 = d_0^2 \left(1 + a^4\gamma^2\right)^k \xrightarrow{k \to \infty} \infty
$$
since $d_0 > 0$ for $(x^0, y^0) \neq (x^\star, y^\star)$ and $a^4\gamma^2 > 0$ for $a \neq 0$, $\gamma > 0$. The distance grows exponentially, $d_k = d_0 \left(1 + a^4\gamma^2\right)^{k/2}$, by a factor $\sqrt{1 + a^4\gamma^2}$ per iteration.

__(Optional $\star$)__ A second attempt at finding the solution via _alternating_ gradient descent ascent on $x$
        and $y$. More precisely for $\gamma > 0$ consider the update
        
\begin{equation}
    x^{k+1} = x^k - \gamma \nabla_x f(x^k, y^k), \qquad
    y^{k+1} = y^k + \gamma \nabla_y f(x^{k+1}, y^k) \nonumber
\end{equation}
Show that the sequence of iterates $\{x^k, y^k \}_{k=0}^\infty$ starting
from any point $(x^0, y^0) \neq (x^\star, y^\star)$ i) never converges, ii) but still remains bounded under certain stepsize conditions.


Note: in this example, for simplicity just consider $f:\mathbb{R}^2 \to \mathbb{R}$ with variables $x,y$, where
$x, y \in \mathbb{R}$, the function has the form of
$
f(x,y) = (a x - b) (a y - c), \quad a \neq 0, \; b,c \in \mathbb{R}.
$


**HINT:** Reduce the problem to studying a linear system.


__(4)__ (10 points) The divergence in (3) relies on three properties of the stepsize $\gamma$: it is positive, it is constant in $k$, and it is the same for both players. Dropping all three makes the very same iteration converge, see https://arxiv.org/abs/2505.01423 for more details. More precisely, allowing each player its own stepsize at each iteration, and allowing negative values, for $\alpha_k, \beta_k \in \mathbb{R}$ consider

$$
x^{k+1} = x^k - \alpha_k \nabla_x f(x^k, y^k), \qquad
y^{k+1} = y^k + \beta_k \nabla_y f(x^k, y^k)
$$

In the three parts below, we will use $z^k = \begin{bmatrix} x^k \\ y^k \end{bmatrix}$ and $z^\star = \begin{bmatrix} x^\star \\ y^\star \end{bmatrix}$, and $d_k = \|z^k - z^\star\|_2$ as in (3).

__(4.a)__ (4 points) Show that the updates above can be written as $z^{k+1} - z^\star = U_k (z^k - z^\star)$, and give the matrix $U_k$. Then show that the iterates fail to converge for every nonnegative schedule ($\alpha_k, \beta_k \geq 0$ for all $k$), for every symmetric schedule ($\alpha_k = \beta_k$ for all $k$), and for every constant schedule $(\alpha_k, \beta_k) \equiv (\alpha, \beta)$. Note that this is stronger than (3), which only ruled out a constant $\gamma > 0$.

**HINT:** Compute $\det U_k$ using the hint of (1). Any $d \times d$ matrix satisfies $\|A\|_2 \geq |\det A|^{1/d}$, so a product of matrices of determinant at least $1$ is never a contraction. The constant case is not covered by this argument; use the eigenvalues of $U$ instead.

Since $\nabla_x f = a^2(y - c/a)$ and $\nabla_y f = a^2(x - b/a)$, we have
$$
\begin{align*}
z^{k+1} - z^\star &= \begin{pmatrix} x^k + \alpha_k a^2(-y^k + c/a) - b/a \\ y^k + \beta_k a^2(x^k - b/a) - c/a \end{pmatrix} \\ \\
\implies U_k &= \begin{bmatrix} I & -a^2\alpha_k I \\ a^2\beta_k I & I \end{bmatrix}, \qquad I \in \mathbb{R}^{2 \times 2}.
\end{align*}
$$
Its characteristic polynomial is
$$
\begin{align*}
\det\left(U_k - \lambda I_4\right) &\overset{\text{hint}}{=} \det\left((1-\lambda)^2 I + a^4\alpha_k\beta_k I\right) \\
&= \left((\lambda - 1)^2 + a^4\alpha_k\beta_k\right)^2
\end{align*}
$$
where the hint applies because all blocks are scalar multiples of $I$. In particular, for $\lambda = 0$,
$$
\det U_k = \left(a^4\alpha_k\beta_k + 1\right)^2 \geq 1
$$
whenever $\alpha_k\beta_k \geq 0$, which holds both for nonnegative schedules and for symmetric ones ($\alpha_k\beta_k = \alpha_k^2$). Then, since $z^k - z^\star = U_{k-1} \cdots U_0 (z^0 - z^\star)$,
$$
\left\|U_{k-1} \cdots U_0\right\|_2 \overset{\text{hint}}{\geq} \left|\det\left(U_{k-1} \cdots U_0\right)\right|^{1/4} = \prod_{j=0}^{k-1} \left|\det U_j\right|^{1/4} \geq 1
$$
so the product is never a contraction and does not tend to $0$: there is always some starting point $z^0$ from which the iterates do not converge to $z^\star$.

For the constant case $U_k \equiv U$, the eigenvalues solve $(\lambda - 1)^2 + a^4\alpha\beta = 0$:
$$
\begin{align*}
\alpha\beta \geq 0 &\implies \lambda = 1 \pm i a^2\sqrt{\alpha\beta}, \qquad |\lambda| = \sqrt{1 + a^4\alpha\beta} \geq 1 \\
\alpha\beta < 0 &\implies \lambda = 1 \pm a^2\sqrt{-\alpha\beta}, \qquad \lambda_+ > 1
\end{align*}
$$
In both cases $|\lambda_{\max}| \geq 1$. Since the matrix is the same at every step, so are its eigenvectors: for an eigenvector $v$ of $\lambda_{\max}$ we get $U^k v = \lambda_{\max}^k v$, hence $\|U^k v\|_2 = |\lambda_{\max}|^k \|v\|_2 \geq \|v\|_2$. Therefore $U^k \not\to 0$ and there is no convergence.

__(4.b)__ (4 points) Fix $h > 0$ and consider the slingshot schedule of https://arxiv.org/abs/2505.01423,

$$
\alpha_{2t} = -\beta_{2t} = -\alpha_{2t+1} = \beta_{2t+1} = h, \qquad t = 0, 1, 2, \dots
$$

under which one of the two players moves in the wrong direction at every iteration. Compute the two-step matrix $U_{2t+1} U_{2t}$, and find for which $h$ the iterates converge and at which rate. Compare with the rate of (3).

With $\alpha_{2t} = -\beta_{2t} = -\alpha_{2t+1} = \beta_{2t+1} = h$, the matrix $U_k$ of (4.a) gives
$$
\begin{align*}
A := U_{2t+1} U_{2t} &= \begin{bmatrix} I & a^2 h I \\ a^2 h I & I \end{bmatrix} \begin{bmatrix} I & -a^2 h I \\ -a^2 h I & I \end{bmatrix} \\
&= \begin{bmatrix} (1 - a^4 h^2) I & 0 \\ 0 & (1 - a^4 h^2) I \end{bmatrix} \\
&= (1 - a^4 h^2) I_4 .
\end{align*}
$$
Then $z^{2t} - z^\star = A^t (z^0 - z^\star) = (1 - a^4h^2)^t (z^0 - z^\star)$, which converges iff
$$
\begin{align*}
\left|1 - a^4 h^2\right| < 1
&\implies h^2 \in \left(0, \frac{2}{a^4}\right) \\
&\implies h \in \left(0, \frac{\sqrt{2}}{a^2}\right)
\end{align*}
$$
where in the last step we used $h > 0$ by hypothesis. The odd iterates also converge, since $z^{2t+1} - z^\star = U_{2t} A^t (z^0 - z^\star)$ and $U_{2t}$ is the same matrix for every $t$, so $\|z^{2t+1} - z^\star\|_2 \leq \|U_{2t}\|_2 \left|1 - a^4h^2\right|^t d_0 \to 0$. Therefore we have convergence for such $h$, with
$$
d_{2t} = \left|1 - a^4h^2\right|^t d_0 .
$$
In (3) we had
$$
d_{2t} = \left(1 + a^4\gamma^2\right)^t d_0 ,
$$
so we have flipped the sign in the two-step factor and obtained linear convergence.

__(4.c)__ (2 points) Let $\Phi(x,y) := \frac{1}{2}\|\nabla f(x,y)\|_2^2$, which vanishes exactly at the saddle point. Show that two slingshot steps amount to one gradient descent step on $\Phi$ with stepsize $h^2$, and that a single update of (3) cannot decrease $\Phi$. Then find the $h$ for which the iterates reach $(x^\star, y^\star)$ exactly, and after how many steps.

**HINT:** $\Phi$ is a quadratic here, write it in terms of $x - x^\star$ and $y - y^\star$. For the second claim, compare the directions $G(z) = \begin{bmatrix} \nabla_x f \\ -\nabla_y f \end{bmatrix}$ and $\nabla \Phi(z)$.

Since $\nabla f = a^2 \begin{pmatrix} y \\ x \end{pmatrix} - a \begin{pmatrix} c \\ b \end{pmatrix}$, we get
$$
\begin{align*}
\Phi(x,y) &\overset{\text{Pythagoras}}{=} \frac{1}{2} a^4 \left( \|y - c/a\|_2^2 + \|x - b/a\|_2^2 \right) = \frac{1}{2} a^4 \left\| \begin{pmatrix} x - x^\star \\ y - y^\star \end{pmatrix} \right\|_2^2 \\ \\
\implies \nabla \Phi(x,y) &= a^4 \begin{pmatrix} x - x^\star \\ y - y^\star \end{pmatrix} .
\end{align*}
$$
A gradient descent step on $\Phi$ with stepsize $h^2$ from $z^{2t}$ gives
$$
\begin{align*}
z^{2t} - h^2 \nabla\Phi(z^{2t}) &= (1 - a^4h^2)\, z^{2t} + a^4h^2\, z^\star \\ \\
\implies z^{2t} - h^2 \nabla\Phi(z^{2t}) - z^\star &= (1 - a^4h^2)(z^{2t} - z^\star)
\end{align*}
$$
which is exactly $z^{2t+2} - z^\star = A(z^{2t} - z^\star)$ from (4.b).

For a single update of (3), let $s^k := z^{k+1} - z^k$. Then
$$
\begin{align*}
\left\langle s^k, \nabla\Phi(z^k) \right\rangle
&= \gamma \left\langle \begin{pmatrix} -\nabla_x f(x^k, y^k) \\ \nabla_y f(x^k, y^k) \end{pmatrix}, \nabla\Phi(x^k, y^k) \right\rangle \\
&= \left\langle \begin{pmatrix} a^2\gamma(-y^k + c/a) \\ a^2\gamma(x^k - b/a) \end{pmatrix}, a^4 \begin{pmatrix} x^k - b/a \\ y^k - c/a \end{pmatrix} \right\rangle \\
&= 0
\end{align*}
$$
since the two terms are opposite to each other. Therefore
$$
\Phi(z^k + s^k) - \Phi(z^k) \overset{\text{Taylor}}{=} \left\langle \nabla\Phi(z^k), s^k \right\rangle + \frac{1}{2} a^4 \|s^k\|_2^2 = \frac{1}{2} a^4 \|s^k\|_2^2 \geq 0
$$
where Taylor's expansion is exact because $\Phi$ is a polynomial of degree $2$, with $\nabla^2\Phi = a^4 I_4$.

For exact convergence, we want
$$
(1 - a^4h^2)^t = 0
$$
which happens iff $1 - a^4h^2 = 0$, iff $h = 1/a^2$. In this case $z^2 = z^\star$, so the iterates reach the saddle point after two steps, i.e. one slingshot pair.

In [ ]:
# (4) Numerical check on the bilinear f above: sGDA vs. the slingshot schedule.
import numpy as np
import matplotlib.pyplot as plt

a, b, c = 1.7, np.array([0.3, -1.1]), np.array([2.0, 0.5])
x_star, y_star = b / a, c / a
grad_x = lambda x, y: a * (a * y - c)
grad_y = lambda x, y: a * (a * x - b)

def run(schedule, n_steps, x0=np.array([3.0, -2.0]), y0=np.array([-1.0, 4.0])):
    """Run GDA with per-iteration stepsizes (alpha_k, beta_k) = schedule(k)."""
    x, y = x0.copy(), y0.copy()
    d = [np.linalg.norm(np.r_[x - x_star, y - y_star])]
    for k in range(n_steps):
        alpha, beta = schedule(k)
        x, y = x - alpha * grad_x(x, y), y + beta * grad_y(x, y)
        d.append(np.linalg.norm(np.r_[x - x_star, y - y_star]))
    return np.array(d)

slingshot = lambda h: (lambda k: (-1)**k * np.array([h, -h]))   # the schedule of (4.b)
h_star = 1/(a**2)                            # the magnitude you found in (4.c)
gamma, h = 0.2, 0.2

d_sgda  = run(lambda k: (gamma, gamma), 20)   # question (3)
d_sling = run(slingshot(h), 20)               # question (4.b)
d_opt   = run(slingshot(h_star), 6)           # question (4.c)

# compare these against the two-step factor and the threshold you derived in (4.b)
print("measured  d_2 / d_0                   :", round(d_sling[2] / d_sling[0], 6))
print("d_k for h = h_star                    :", d_opt.round(12))

plt.style.use('seaborn-v0_8-poster')
plt.figure(figsize=(9, 5.5))
plt.semilogy(d_sgda, lw=4, label=rf"sGDA, $\gamma = {gamma}$")
plt.semilogy(d_sling, lw=4, label=rf"slingshot, $h = {h}$")
plt.semilogy(np.maximum(d_opt, 1e-18), lw=4, ls="--", marker="o", ms=12,
             label=r"slingshot, $h = h^\star$")
plt.xlabel("$k$")
plt.ylabel("$d_k$")
plt.legend()
plt.tight_layout()
plt.show()

### 1.2 A concrete example: Rock - Paper - Scissors - Lizard - Spock - 35 points

To make your previous abstract analysis more concrete, we will now look at the iconic Rock-Paper-Scissors-Lizard-Spock game. Sheldon and Leonard always like to settle their differences through a round of this game, whose rules they state as follows:

*"Scissors cuts Paper, Paper covers Rock, Rock crushes Lizard, Lizard poisons Spock, Spock smashes Scissors, Scissors decapitates Lizard, Lizard eats Paper, Paper disproves Spock, Spock vaporizes Rock, (and as it always has) Rock crushes Scissors"*

Sheldon and Leonard always pick to play Spock because "logic trumps all", it will be your task to show that this is not the optimal strategy.

We can formalize the game as follows. There are two players, the `x` player (e.g. Sheldon) and the `y` player (e.g. Leonard). There is only one single round. The players play a randomized strategy: each player chooses a probability of playing rock/paper/scissors/lizard/Spock. We look at the expected pay-off of these randomized strategies. 

In other words, the players choose an element in $\Delta_5$ the probability simplex in dimension 5. The `x` player chooses a vector $\mathbf{x} = \begin{bmatrix} \mathbf{x}_1 & \mathbf{x}_2 & \mathbf{x}_3 & \mathbf{x}_4 & \mathbf{x}_5 \end{bmatrix} \in \Delta_5$ where $\mathbf{x}_1$ is the probability of playing `Rock`, $\mathbf{x}_2$ is the probability of playing `Paper`, $\mathbf{x}_3$ is the probability of playing `Scissors`, $\mathbf{x}_4$ is the probability of playing `Lizard`, $\mathbf{x}_5$ is the probability of playing `Spock`. The `y` player chooses a vector $\mathbf{y} \in \Delta_5$ defined in the same way. 

The game designers decide that winning the game gives 1 point and a tie gives 0 points. So the expected payoff for a give choice of strategies $\mathbf{x}, \mathbf{y}$ is obtained by computing:
$$
\mathbf{x}^\top \begin{bmatrix} 0 & 1 & -1 & -1 & 1 \\ -1 & 0 & 1 & 1 & -1 \\ 1 & -1 & 0 & -1 & 1 \\ 1 & -1 & 1 & 0 & -1 \\ -1 & 1 & -1 & 1 & 0 \end{bmatrix}\mathbf{y}
$$

The `x` player wants to minimize this expected payoff and the `y` player wants to maximize it. So the problem we seek to solve is

$$
\min_{\mathbf{x} \in \Delta_5} \max_{\mathbf{y} \in \Delta_5} \mathbf{x}^\top \mathbf{M}\mathbf{y} =: f(\mathbf{x}, \mathbf{y})
$$
with $\mathbf{M} = \begin{bmatrix} 0 & 1 & -1 & -1 & 1 \\ -1 & 0 & 1 & 1 & -1 \\ 1 & -1 & 0 & -1 & 1 \\ 1 & -1 & 1 & 0 & -1 \\ -1 & 1 & -1 & 1 & 0 \end{bmatrix}$.

In the following cells, you will implement methods to solve this game and find the optimal strategies.

In [ ]:
import torch
from ipywidgets import interact, SelectionSlider, fixed
from itertools import combinations
from scipy.stats import entropy
from lib.utils import *

**(1)** (2 points) Implement the objective function $f$ given two 3 dimensional vectors `x` and `y` stored as `torch.Tensor` objects.

In [ ]:
M = torch.tensor([[0, 1, -1, -1, 1], [-1, 0, 1, 1, -1], [1, -1, 0, -1, 1], [1, -1, 1, 0, -1], [-1, 1, -1, 1, 0]], dtype=torch.float32)
def f(x: torch.Tensor, y: torch.Tensor):
    return x @ M @ y

**(2)** (10 points) Implement a single step of the simultaneous gradient descent/ascent studied in question 1.1.(3). Since we are in a constrained setting, make sure you include a projection step onto the simplex. 

The function `GDA` you will implement takes in the objective function `f`, the two current strategies of the players stored in tensors `x` and `y` and a step_size. Write the function so that it modifies the variables `x` and `y` in place without returning anything.

We provide you with a function called `simplex_project` that projects a vector on the probability simplex. The function has no return value and does the projection in place. Use `Pytorch` to compute gradients automatically. 

**HINT:** Review what a call to `.backward()` on a tensor does. Review what in-place operations are like `.add_`. Think of which steps need to be in a `with torch.no_grad()` block. Remember to zero the gradients before re-using them.

In [ ]:
def GDA(f, x, y, step_size):
    payoff = f(x, y)
    payoff.backward()

    with torch.no_grad():
        x.add_(-step_size * x.grad)
        y.add_(step_size * y.grad)
        simplex_project(x)
        simplex_project(y)
        
    x.grad.zero_()
    y.grad.zero_()

We can run this algorithm initialized from Sheldon and Leonard's strategies of always playing Spock.

In [ ]:
x_init = torch.tensor([0.0, 0.0, 0.0, 0.0, 1.0])
y_init = torch.tensor([0.0, 0.0, 0.0, 0.0, 1.0])

gda_x_sequence, gda_y_sequence = run_alg(GDA, f, x_init, y_init, step_size=0.05, n_iterations=2500)

We can visualize this sequence as ${4 \choose 2} = 6$ two dimensional slices since the simplex $\Delta_5$ is 4-dimensional.

In [ ]:
dim_pairs = list(combinations(range(4), 2))
def visualize_seq_slice(dim_pair):
    visualize_seq(gda_x_sequence[:, dim_pairs[dim_pair]], 
                  gda_y_sequence[:, dim_pairs[dim_pair]],
                  dim_pairs[dim_pair])
interact(visualize_seq_slice, dim_pair=SelectionSlider(
    options=range(len(dim_pairs)),
    value=0,
    description="Dim pair:",
    disabled=False,
    continuous_update=False,
    orientation="horizontal",
    readout=True
))

To have plots that aggregate the convergence status over all 5 dimensions at the same time, let's also visualize:

- the duality gap over time: $g_k = \max_{\mathbf{y} \in \Delta_5}{f(\mathbf{x}_k,\mathbf{y})} - \min_{\mathbf{x} \in \Delta_5}{f(\mathbf{x},\mathbf{y}_k)} = \max_{i}{(\mathbf{M}^\top \mathbf{x}_k)_i} - \min_{i}{(\mathbf{M} \mathbf{y}_k)_i}$
- the distance to the optimum over time: $d_k=\sqrt{||\mathbf{x}_k-\mathbf{x}^\star||_2^2 + ||\mathbf{y}_k-\mathbf{y}^\star||_2^2}$
- the discrete entropy over time of the x and y strategies (we can do this since they are probability distributions): $H(\mathbf{x}_k)=-\sum_{i=1}^{5}{\mathbf{x}_k^{(i)}\log{\mathbf{x}_k^{(i)}}}, H(\mathbf{y}_k)=-\sum_{i=1}^{5}{\mathbf{y}_k^{(i)}\log{\mathbf{y}_k^{(i)}}}$

In [ ]:
def distance_to_opt(L_x, L_y):
    return np.sqrt(np.sum((L_x - 0.2) ** 2 + (L_y - 0.2) ** 2, axis=1))

In [ ]:
def duality_gap(L_x, L_y):
    return (np.max(M.numpy().T.reshape(1, 5, 5) @ L_x.reshape(-1, 5, 1), axis=(1, 2)) 
            - np.min(M.numpy().reshape(1, 5, 5) @ L_y.reshape(-1, 5, 1), axis=(1, 2)))

In [ ]:
plt.style.use('seaborn-v0_8-poster')
fig, ax = plt.subplots(3, 1, figsize=(11.7, 1.5 * 8.3))
ax[0].plot(range(len(gda_x_sequence)), entropy(gda_x_sequence, axis=1), lw=5, color='b', label="GDA x")
ax[0].plot(range(len(gda_y_sequence)), entropy(gda_y_sequence, axis=1), lw=5, color='r', label="GDA y")
ax[0].axline((0, np.log(5)), slope=0, color='black', lw=5, label="Max")
ax[0].legend()
ax[0].set_xscale("log")
ax[0].set_xlabel("k")
ax[0].set_ylabel("Entropy")
ax[1].plot(range(len(gda_x_sequence)), distance_to_opt(gda_x_sequence, gda_y_sequence), lw=5, label="GDA")
ax[1].legend()
ax[1].set_yscale("log")
ax[1].set_xlabel("k")
ax[1].set_ylabel("$d_k$")
ax[2].plot(range(len(gda_x_sequence)), duality_gap(gda_x_sequence, gda_y_sequence), lw=5, label="GDA")
ax[2].legend()
ax[2].set_yscale("log")
ax[2].set_xlabel("k")
ax[2].set_ylabel("$g_k$")
fig.tight_layout()
plt.show()

In [ ]:
step_sizes = [0.005, 0.01, 0.05, 0.1, 0.5]
n_iter = 10000
plt.style.use('seaborn-v0_8-poster')
for step in step_sizes:
    L_x, L_y = run_alg(GDA, f, x_init, y_init, step_size=step, n_iterations=n_iter)
    fig, ax = plt.subplots(3, 1, figsize=(11.7, 1.5 * 8.3))
    ax[0].plot(range(len(L_x)), entropy(L_x, axis=1), lw=5, color='b', label="GDA x")
    ax[0].plot(range(len(L_y)), entropy(L_y, axis=1), lw=5, color='r', label="GDA y")
    ax[0].axline((0, np.log(5)), slope=0, color='black', lw=5, label="Max")
    ax[0].legend()
    ax[0].set_xscale("log")
    ax[0].set_xlabel("k")
    ax[0].set_ylabel("Entropy")
    ax[0].set_title(f"step_size = {step}")
    ax[1].plot(range(len(L_x)), distance_to_opt(L_x, L_y), lw=5, label="GDA")
    ax[1].legend()
    ax[1].set_yscale("log")
    ax[1].set_xlabel("k")
    ax[1].set_ylabel("$d_k$")
    ax[2].plot(range(len(L_x)), duality_gap(L_x, L_y), lw=5, label="GDA")
    ax[2].legend()
    ax[2].set_yscale("log")
    ax[2].set_xlabel("k")
    ax[2].set_ylabel("$g_k$")
    fig.tight_layout()
    plt.show()

**(3)** (3 point) What kind of behavior do you observe ? Do the iterates converge ? Play with the step_size and the number of iterations.




The iterates never converge, whatever step size we use. With step size $0.05$, $d_k$ first drops to about $0.4$ but then settles around $0.5$, $g_k$ keeps oscillating around $1.2$, and the entropy stays around $1.2$, below the maximum $\log 5 \approx 1.6$, so the players never reach the uniform strategy. The $x$ and $y$ curves overlap because $M^\top = -M$ and both players start from Spock, so their updates are identical.

Changing the step size or running for $10000$ iterations only changes how the iterates oscillate. Large step sizes reach the oscillating regime immediately but farther from the optimum (with $0.5$, $d_k \approx 0.8$ and $g_k$ gets close to $2$, its maximum). Small step sizes get closer at first ($d_k \approx 0.3$), but then $d_k$ slowly grows again. This matches 1.1.(3): without constraints sGDA spirals away from the saddle point, and for small step sizes this happens slowly. Here the projection onto the simplex keeps the iterates bounded, so they keep cycling at a roughly constant distance from $(x^\star, y^\star)$ instead of diverging. A smaller step size only slows this down, it does not make sGDA converge.

**(4)** (4 points) As you've shown both theoretically and in practice, simultaneous Gradient Descent Ascent (sGDA) cannot solve the problem. You will now implement algorithm that can solve bilinear games. We will denote the projection onto the decision space of the x-player and y-player, $\Pi_{\mathcal X}$ and $\Pi_{\mathcal Y}$, respectively.


Let $\mathbf{z}_k = \begin{bmatrix} x_k \\ y_k \end{bmatrix}$, $\Pi(\mathbf{z})=\begin{bmatrix} \Pi_{\mathcal X}(x) \\ \Pi_{\mathcal Y}(y) \end{bmatrix}$ and $G(\mathbf{z}_k) = \begin{bmatrix} \nabla_x f(x_k,y_k) \\ -\nabla_y f(x_k,y_k) \end{bmatrix}$.

Consider the following implicit updates, which is the `Proximal Point Method ` (PPM) for solving the bilinear games:
\begin{equation}
    \mathbf{z}_{k+1} = \Pi(\mathbf{z}_k - \gamma G({{\color{red}{\mathbf{z}_{k+1}}}})) \nonumber
\end{equation}


Let’s first prove that the Proximal Point Method (PPM) converges in a simple case to build intuition. Consider the aforementioned stylized function $f: \mathbb{R}^2 \to \mathbb{R}$, such that $f(x, y)=xy$, and consider simplified unconstrained probelm, i.e., the projection is the identity map. Prove that PPM converges.

**HINT:** Rewrite the implicit update as an explicit update in the form of $z_{k+1} = A z_{k}$ for some matrix $A$.

**HINT:** A square matrix $A$ satisfies $\lim_{k \to \infty} A^k = 0$ if its spectral radius $\rho(A) < 1$, where the spectral radius is defined by
$
\rho(A) := \max_i |\lambda_i|,
$
with $\{\lambda_i\}$ denoting the  eigenvalues of $A$.

Since $\nabla f = \begin{pmatrix} y \\ x \end{pmatrix}$, the PPM update reads
$$
\begin{align*}
\begin{pmatrix} x_{k+1} \\ y_{k+1} \end{pmatrix} &= \begin{pmatrix} x_k \\ y_k \end{pmatrix} - \gamma \begin{pmatrix} y_{k+1} \\ -x_{k+1} \end{pmatrix} \\ \\
\implies \begin{pmatrix} x_k \\ y_k \end{pmatrix} &= \underbrace{\begin{pmatrix} 1 & \gamma \\ -\gamma & 1 \end{pmatrix}}_{=: B} \begin{pmatrix} x_{k+1} \\ y_{k+1} \end{pmatrix} .
\end{align*}
$$
$B$ is invertible, since $\det B = 1 + \gamma^2 \neq 0$, so we have $z_{k+1} = A z_k$ with
$$
A = B^{-1} = \frac{1}{1 + \gamma^2} \begin{pmatrix} 1 & -\gamma \\ \gamma & 1 \end{pmatrix} .
$$
It's easier to study the eigenvalues of $B$:
$$
\begin{align*}
\det(B - \lambda I) &= (\lambda - 1)^2 + \gamma^2 \\ \\
\implies \lambda^B_{1,2} &= 1 \pm i\gamma \implies \left|\lambda^B_{1,2}\right| = \sqrt{1 + \gamma^2} > 1 \\ \\
\implies \left|\lambda^A_{1,2}\right| &\overset{(i)}{=} \frac{1}{\left|\lambda^B_{1,2}\right|} < 1
\end{align*}
$$
where (i) holds because $A = B^{-1}$, so $\lambda^A_{1,2} = 1/\lambda^B_{1,2}$. Therefore $\rho(A) < 1$, hence $A^k \to 0$ and $z_k = A^k z_0 \to 0$, i.e. PPM converges to the saddle point $(x^\star, y^\star) = (0, 0)$ for every $\gamma > 0$.

**(5)** (6 points)  In general, computing the next iterate $\mathbf{z}_{k+1}$ in PPM requires you to solve a fixed point problem because we are evaluating the gradient at the unknown next iterate. As this is too costly, we circumvent this difficulty by doing an _extrapolation_ step. The idea behind `ExtraGradient` (EG) is to approximate an implicit update with a more tractable one. Define the half steps:

\begin{equation}
    \mathbf{z}_{k+1/2} = \Pi(\mathbf{z}_k - \gamma G(\mathbf{z}_k)) \nonumber
\end{equation}

These half step extrapolation variables will help us to approximate the implicit iterates. We can then write
\begin{equation}
    \mathbf{z}_{k+1} = \Pi(\mathbf{z}_k - \gamma G({{\color{green}{\mathbf{z}_{k+1/2}}}})) \nonumber
\end{equation}
The recursion above defines the ExtraGradient algorithm.

In [ ]:
def ExtraGradient(f, x, y, step_size):
    payoff = f(x, y)
    payoff.backward()
    with torch.no_grad():
        x_k_half = x - step_size * x.grad
        y_k_half = y + step_size * y.grad
        simplex_project(x_k_half)
        simplex_project(y_k_half)
            
    x_k_half = x_k_half.detach().requires_grad_(True)
    y_k_half = y_k_half.detach().requires_grad_(True)
    payoff_half = f(x_k_half, y_k_half)
    payoff_half.backward()

    with torch.no_grad():
        x.add_( -step_size * x_k_half.grad)
        y.add_(  step_size * y_k_half.grad)
        simplex_project(y)
        simplex_project(x)

    x.grad.zero_()
    y.grad.zero_()

In [ ]:
eg_x_sequence, eg_y_sequence = run_alg(ExtraGradient, f, x_init, y_init, step_size=0.1, n_iterations=2500)

In [ ]:
dim_pairs = list(combinations(range(4), 2))
def visualize_seq_slice(dim_pair):
    visualize_seq(eg_x_sequence[:, dim_pairs[dim_pair]], 
                  eg_y_sequence[:, dim_pairs[dim_pair]],
                  dim_pairs[dim_pair])
interact(visualize_seq_slice, dim_pair=SelectionSlider(
    options=range(len(dim_pairs)),
    value=0,
    description="Dim pair:",
    disabled=False,
    continuous_update=False,
    orientation="horizontal",
    readout=True
))

In [ ]:
plt.style.use('seaborn-v0_8-poster')
fig, ax = plt.subplots(3, 1, figsize=(11.7, 1.5 * 8.3))
ax[0].plot(range(len(gda_x_sequence)), entropy(gda_x_sequence, axis=1), lw=5, color='b', alpha=0.1, label="GDA x")
ax[0].plot(range(len(gda_y_sequence)), entropy(gda_y_sequence, axis=1), lw=5, color='r', alpha=0.1, label="GDA y")
ax[0].plot(range(len(eg_x_sequence)), entropy(eg_x_sequence, axis=1), lw=5, color='b', label="EG x")
ax[0].plot(range(len(eg_y_sequence)), entropy(eg_y_sequence, axis=1), lw=5, color='r', label="EG y")
ax[0].axline((0, np.log(5)), slope=0, color='black', lw=5, alpha=0.5, label="Max")
ax[0].legend()
ax[0].set_xscale("log")
ax[0].set_xlabel("k")
ax[0].set_ylabel("Entropy")
ax[1].plot(range(len(gda_x_sequence)), distance_to_opt(gda_x_sequence, gda_y_sequence), lw=5, label="GDA")
ax[1].plot(range(len(eg_x_sequence)), distance_to_opt(eg_x_sequence, eg_y_sequence), lw=5, label="EG")
ax[1].legend()
ax[1].set_yscale("log")
ax[1].set_xlabel("k")
ax[1].set_ylabel("$d_k$")
ax[2].plot(range(len(gda_x_sequence)), duality_gap(gda_x_sequence, gda_y_sequence), lw=5, label="GDA")
ax[2].plot(range(len(eg_x_sequence)), duality_gap(eg_x_sequence, eg_y_sequence), lw=5, label="EG")
ax[2].legend()
ax[2].set_yscale("log")
ax[2].set_xlabel("k")
ax[2].set_ylabel("$g_k$")
fig.tight_layout()
plt.show()

**(BONUS)** What can you observe about the optimal solution? What properties does the optimal Rock-Paper-Scissors-Lizard-Spock strategy have? Prove that extra-gradient in the bilinear case doesn't diverge like GDA. (Again, candy reward for correct answers!)




Observations: EG converges, unlike GDA. In the log-scale plots $d_k$ and $g_k$ go down along a straight line, so the convergence is linear, and after $2500$ iterations both are below $10^{-3}$, while for GDA they stay around $0.5$ and above $1$. The entropy of $x$ and $y$ keeps growing until it reaches the black line $\log 5$, after about $1000$ iterations, which means that both players end up playing every action with the same probability, $x^\star = y^\star = \frac{1}{5}\mathbf{1}$.

Optimal strategy:
- It is uniform, which is why its entropy is the maximum $\log 5$, and it is the same for both players.
- The value of the game is $0$, so the game is fair. Since $M^\top = -M$, $f(x,x) = x^\top M x = 0$, so for any $x$ the $y$ player can get at least $0$ by playing $y = x$. On the other hand every row of $M$ has two $+1$ and two $-1$, so $M\mathbf{1} = 0$ and against $x^\star$ every $y$ gets exactly $0$.
- For the same reason every action gets the same expected payoff against the uniform strategy, so the opponent cannot exploit it. Always playing Spock instead loses against Paper and Lizard, so Sheldon and Leonard are wrong.

EG does not diverge: we use the same $f$ and notation as in 1.1.(3), with $u^k = x^k - x^\star$ and $v^k = y^k - y^\star$, so that $\nabla_x f = a^2 v$ and $\nabla_y f = a^2 u$. The extrapolation step gives $u^{k+1/2} = u^k - \gamma a^2 v^k$ and $v^{k+1/2} = v^k + \gamma a^2 u^k$, and then
$$
\begin{align*}
u^{k+1} &= u^k - \gamma a^2 v^{k+1/2} = (1 - \gamma^2 a^4)\, u^k - \gamma a^2 v^k, \\
v^{k+1} &= v^k + \gamma a^2 u^{k+1/2} = (1 - \gamma^2 a^4)\, v^k + \gamma a^2 u^k .
\end{align*}
$$
Expanding the squared norms like in (3), the cross terms cancel again and we get
$$
d_{k+1}^2 = \left( (1 - \gamma^2 a^4)^2 + \gamma^2 a^4 \right) d_k^2 = \left( 1 - \gamma^2 a^4 + \gamma^4 a^8 \right) d_k^2 .
$$
The factor is smaller than $1$ for $0 < \gamma < 1/a^2$, so EG converges linearly, and for $\gamma = 1/a^2$ the distance stays the same. With GDA the factor was $1 + \gamma^2 a^4$, always bigger than $1$: the extrapolation changes the sign of the $\gamma^2 a^4$ term and only adds a $\gamma^4 a^8$ term, which is small when $\gamma$ is small. The straight line of $d_k$ in the log plot is the linear convergence we expect from this formula, and it shows that $\gamma = 0.1$ is small enough for our game.

**(6)** (5 points) It turns out that we can _generalize_ the ExtraGradient method naturally by striving for a bit more accurate extrapolation. Namely, instead of always performing 1 extrapolation step updating half-iterates before the main parameter update, we can perform $m\geq 1$ fractional steps. Implement this ClairvoyantExtraGradient (CEG) method [1], with the precise update step formulas given below:

$$
\begin{aligned}
\mathbf{z}_{k+{\color{green}{\tfrac{1}{m+1}}}} &= \Pi\big(\mathbf{z}_k - \gamma G({\color{green}{\mathbf{z}_{k}}})\big) \\
\mathbf{z}_{k+{\color{green}{\tfrac{2}{m+1}}}} &= \Pi\big(\mathbf{z}_{k+{\color{green}{\tfrac{1}{m+1}}}} - \gamma G(\mathbf{z}_{k+{\color{green}{\tfrac{1}{m+1}}}})\big) \\
&\;\;\vdots \\
\mathbf{z}_{k+{\color{green}{\tfrac{m}{m+1}}}} &= \Pi\big(\mathbf{z}_{k+{\color{green}{\tfrac{m-1}{m+1}}}} - \gamma G(\mathbf{z}_{k+{\color{green}{\tfrac{m-1}{m+1}}}})\big) \\
\mathbf{z}_{k+1} &= \Pi\big(\mathbf{z}_k - \gamma G(\mathbf{z}_{k+{\color{green}{\tfrac{m}{m+1}}}})\big)
\end{aligned}
$$


[1] Cevher, Volkan, Georgios Piliouras, Ryann Sim, and Stratis Skoulakis. “Min-Max Optimization Made Simple: Approximating the Proximal Point Method via Contraction Maps.” In 2023 Symposium on Simplicity in Algorithms (SOSA), 192–206. Proceedings. Society for Industrial and Applied Mathematics, 2023. https://doi.org/10.1137/1.9781611977585.ch18.

In [ ]:
def ClairvoyantExtraGradient(f, x, y, step_size, m=1):
    payoff = f(x, y)
    x_m = x.detach().clone()
    y_m = y.detach().clone()
    for i in range(m):
        x_m = x_m.detach().requires_grad_(True)
        y_m = y_m.detach().requires_grad_(True)

        payoff = f(x_m, y_m)
        payoff.backward()

        with torch.no_grad():
            x_m.add_( - step_size * x_m.grad)
            y_m.add_(step_size * y_m.grad)
            simplex_project(x_m)
            simplex_project(y_m)

        x_m.grad.zero_()
        y_m.grad.zero_()

    x_m = x_m.detach().requires_grad_(True)
    y_m = y_m.detach().requires_grad_(True)
    payoff = f(x_m, y_m)
    payoff.backward()
    
    with torch.no_grad():
        x.add_( - step_size * x_m.grad)
        y.add_( step_size * y_m.grad)
        simplex_project(x)
        simplex_project(y)
    x_m.grad.zero_()
    y_m.grad.zero_()

In [ ]:
m_values = [1, 2, 3, 5, 10, 25] # You can change this if you wish
ceg_x_sequences, ceg_y_sequences = [], []
for m in m_values:
    print("m =", m)
    step_size = 0.1 if m != 10 else 0.09 # You can change this if you wish
    ceg_x_sequence, ceg_y_sequence = run_alg(ClairvoyantExtraGradient, f, x_init, y_init, 
                                             n_iterations=2500, step_size=step_size, m=m)
    ceg_x_sequences.append(ceg_x_sequence)
    ceg_y_sequences.append(ceg_y_sequence)

In [ ]:
dim_pairs = list(combinations(range(4), 2))
def visualize_seq_slice(L_x, L_y, dim_pair):
    visualize_seq(L_x[:, dim_pairs[dim_pair]], 
                  L_y[:, dim_pairs[dim_pair]],
                  dim_pairs[dim_pair])
for m, ceg_x_sequence, ceg_y_sequence in zip(m_values, ceg_x_sequences, ceg_y_sequences):
    print("m =", m)
    interact(visualize_seq_slice, L_x=fixed(ceg_x_sequence), L_y=fixed(ceg_y_sequence), dim_pair=SelectionSlider(
        options=range(len(dim_pairs)),
        value=0,
        description="Dim pair:",
        disabled=False,
        continuous_update=False,
        orientation="horizontal",
        readout=True
    ))

In [ ]:
plt.style.use('seaborn-v0_8-poster')
fig, ax = plt.subplots(3, 1, figsize=(11.7, 1.5 * 8.3))
ax[0].plot(range(len(gda_x_sequence)), entropy(gda_x_sequence, axis=1), lw=5, alpha=0.1, label="GDA x")
ax[1].plot(range(len(gda_x_sequence)), distance_to_opt(gda_x_sequence, gda_y_sequence), lw=5, label="GDA")
ax[2].plot(range(len(gda_x_sequence)), duality_gap(gda_x_sequence, gda_y_sequence), lw=5, label="GDA")
for m, ceg_x_sequence, ceg_y_sequence in zip(m_values, ceg_x_sequences, ceg_y_sequences):
    ax[0].plot(range(len(ceg_x_sequence)), entropy(ceg_x_sequence, axis=1), lw=5, label=f"CEG-{m} x")
    ax[1].plot(range(len(ceg_x_sequence)), distance_to_opt(ceg_x_sequence, ceg_y_sequence), lw=5, label=f"CEG-{m}")
    ax[2].plot(range(len(ceg_x_sequence)), duality_gap(ceg_x_sequence, ceg_y_sequence), lw=5, label=f"CEG-{m}")
ax[0].axline((0, np.log(5)), slope=0, color='black', lw=5, alpha=0.5, label="Max")
ax[0].legend()
ax[0].set_xscale("log")
ax[0].set_xlabel("k")
ax[0].set_ylabel("Entropy")
ax[1].legend()
ax[1].set_yscale("log")
ax[1].set_xlabel("k")
ax[1].set_ylabel("$d_k$")
ax[2].legend()
ax[2].set_yscale("log")
ax[2].set_xlabel("k")
ax[2].set_ylabel("$g_k$")
fig.tight_layout()
plt.show()
fig.tight_layout()
plt.show()

**(7)** (5 points) Discuss what you observe for the CEG runs. How do the value of $m$ and the step size influence the convergence and the computation cost? Is there a sweet spot?




In [ ]:
tol = 1e-6

def first_iteration_below(values, tol):
    # first iteration k with values[k] <= tol, None if it never happens
    for k, value in enumerate(values):
        if value <= tol:
            return k
    return None

plt.figure(figsize=(11.7, 6))
for m, L_x, L_y in zip(m_values, ceg_x_sequences, ceg_y_sequences):
    d = distance_to_opt(L_x, L_y)
    grad_evals = (m + 1) * np.arange(len(d))
    plt.semilogy(grad_evals, d, lw=4, label=f"CEG-{m}")

    k = first_iteration_below(d, tol)
    if k is None:
        print(f"m = {m:2d}: d_k never below {tol} in {len(d) - 1} iterations ({grad_evals[-1]} gradient evaluations)")
    else:
        print(f"m = {m:2d}: d_k below {tol} after {k} iterations = {grad_evals[k]} gradient evaluations")
plt.xlim(0, 8000)
plt.xlabel("gradient evaluations")
plt.ylabel("$d_k$")
plt.legend()
plt.show()

In [ ]:
# effect of the step size: every m of the template, each with the same list of step sizes.
step_sizes = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3]

plt.figure(figsize=(11.7, 6))
for m in m_values:
    print(f"m = {m}")
    converged_steps = []
    converged_costs = []
    for step in step_sizes:
        L_x, L_y = run_alg(ClairvoyantExtraGradient, f, x_init, y_init, n_iterations=2500, step_size=step, m=m)
        d = distance_to_opt(L_x, L_y)
        k = first_iteration_below(d, tol)
        if k is None:
            print(f"   step size {step}: not below {tol} in 2500 iterations, final d_k = {d[-1]:.1e}")
        else:
            print(f"   step size {step}: below {tol} after {k} iterations = {k * (m + 1)} gradient evaluations")
            converged_steps.append(step)
            converged_costs.append(k * (m + 1))
    plt.plot(converged_steps, converged_costs, marker="o", ms=12, lw=4, label=f"CEG-{m}")
plt.yscale("log")
plt.xlabel("step size")
plt.ylabel("gradient evaluations to get $d_k < 10^{-6}$")
plt.legend()
plt.show()

All the CEG runs converge linearly (straight lines in the log plots), unlike GDA. A larger $m$ needs fewer iterations: $m=25$ reaches the floor just below $10^{-6}$ (float32 precision) after about $200$ iterations, $m=5$ after about $600$ and $m=2$ after about $1800$, while $m=1$ (plain EG) is only slightly below $10^{-3}$ at $k=2500$.

However, each iteration costs $m+1$ gradient evaluations ($m$ extrapolation steps plus the final update), so fewer iterations does not mean less computation. Counting gradient evaluations (first plot above), $m=5$ is the cheapest, with about $3400$ to get $d_k$ below $10^{-6}$, better than $m=10$ ($\approx 3700$) and clearly better than $m=25$ ($\approx 4600$), $m=2$ ($\approx 5000$) and $m=1$, which is not there yet after $5000$. So with the step sizes used above the sweet spot is around $m=5$.

The step size matters even more, as the grid in the last cell shows. For every $m$ a larger step reduces the cost until the method stops converging, and this limit gets smaller as $m$ grows: $0.3$ still works for $m=1$ and $m=2$, the limit is $0.25$ for $m=3$ and $0.15$ for $m=5$, while $m=10$ and $m=25$ converge only for a few isolated values. With the step tuned for each $m$, the cheapest runs are $m=2$ with $0.3$ and $m=3$ with $0.25$, five to six times less than $m=5$ with $0.1$. So the real sweet spot is a small $m$ with the largest step size that still converges.

__(Optional $\star$)__ Apply the slingshot schedule of 1.1.(4) to the game of this section. Implement `SlingshotGDA`, in which one call performs one slingshot pair, i.e. the two steps $(\alpha, \beta) = (h, -h)$ and then $(\alpha, \beta) = (-h, h)$, each followed by a projection onto the simplex. One call then costs two gradient evaluations, like one `ExtraGradient` call. Compare $d_k$ and $g_k$ against GDA and EG at an equal number of gradient evaluations, and check numerically the largest usable $h$ against the threshold of 1.1.(4.b) with $a^2$ replaced by $\sigma_{\max}(\mathbf{M})$.

Note: the analysis of 1.1.(4), like that of the paper it links to, is unconstrained, and the cancellation of (4.b) breaks as soon as a projection clips an iterate, so what you observe here is not covered by the theory.

**HINT:** The structure is that of `GDA`, looped over the two $(\alpha, \beta)$ pairs; remember to recompute the gradients between the two steps. `run_alg` records one point per call, so `n_iterations` calls cover $2\times$`n_iterations` gradient steps.

In [ ]:
def SlingshotGDA(f, x, y, step_size):
    # One call = one slingshot pair = two gradient steps.
    for alpha, beta in [(step_size, -step_size), (-step_size, step_size)]:
        payoff = f(x, y)
        payoff.backward()          # gradients at the current point, recomputed at every step
        with torch.no_grad():
            x.add_(-alpha * x.grad)
            y.add_(beta * y.grad)
            simplex_project(x)
            simplex_project(y)
        x.grad.zero_()
        y.grad.zero_()


In [ ]:
# 1250 calls = 2500 gradient steps, the same budget as the GDA and EG runs above.
sling_x_sequence, sling_y_sequence = run_alg(SlingshotGDA, f, x_init, y_init,
                                             step_size=0.4, n_iterations=1250)

In [ ]:
plt.style.use('seaborn-v0_8-poster')
fig, ax = plt.subplots(2, 1, figsize=(11.7, 8.3))
# x-axis is the number of gradient evaluations: 1 per GDA step, 2 per EG / slingshot call.
grads = {"GDA": np.arange(len(gda_x_sequence)),
         "EG": 2 * np.arange(len(eg_x_sequence)),
         "Slingshot GDA": 2 * np.arange(len(sling_x_sequence))}
seqs = {"GDA": (gda_x_sequence, gda_y_sequence),
        "EG": (eg_x_sequence, eg_y_sequence),
        "Slingshot GDA": (sling_x_sequence, sling_y_sequence)}
for name, (L_x, L_y) in seqs.items():
    ax[0].plot(grads[name], distance_to_opt(L_x, L_y), lw=5, label=name)
    ax[1].plot(grads[name], duality_gap(L_x, L_y), lw=5, label=name)
ax[0].set_ylabel("$d_k$")
ax[1].set_ylabel("$g_k$")
for a_ in ax:
    a_.set_yscale("log")
    a_.set_xlabel("gradient evaluations")
    a_.legend()
fig.tight_layout()
plt.show()

In [ ]:
# (ii) check the threshold you predicted in 1.1.(4.b) against the singular values of M
threshold = np.sqrt(2) / np.linalg.norm(M.numpy(), ord=2)
print(f"sigma_max(M) = {np.linalg.svd(M.numpy())[1][0]:.4f},  predicted threshold = {threshold:.4f}\n")
for h in [0.1, 0.2, 0.3, 0.4, 0.45, 0.46, 0.5, 0.6]:
    L_x, L_y = run_alg(SlingshotGDA, f, x_init, y_init, step_size=h, n_iterations=1250)
    print(f"h = {h:.2f}   d_final = {distance_to_opt(L_x, L_y)[-1]:.3e}")

At the same number of gradient evaluations the slingshot schedule with $h=0.4$ is much faster than both GDA and EG: $d_k$ and $g_k$ reach the floor below $10^{-6}$ after less than $500$ gradient evaluations, while after $2500$ EG is still around $10^{-2}$ and GDA does not go down at all ($d_k \approx 0.5$, $g_k$ above $1$).. The threshold of 1.1.(4.b) with $a^2$ replaced by $\sigma_{\max}(M) \approx 3.08$ gives $h < \sqrt{2}/\sigma_{\max}(M) \approx 0.46$, and the check agrees: for every $h$ up to $0.45$ the final $d_k$ is below $10^{-3}$, while from $h=0.46$ on it is around $0.5$–$0.8$, so the largest usable $h$ is right at the predicted threshold.